# Chapter 5 diagnostic study

Colab-ready notebook wrapper for the ResNet-50 / CLIP / Mistral feature-alignment pipeline.

**Runtime:** Select **Runtime → Change runtime type → T4 GPU** (or better) before running extraction. Extraction downloads COCO and model weights, so it can take a while.

## 1. Install dependencies

Run once per fresh Colab runtime.

In [ ]:
%pip install -q transformers accelerate bitsandbytes POT scikit-learn matplotlib torchvision pillow

## 2. Create the cleaned Python script

This keeps the original command-line pipeline intact while making the workflow easy to run from Colab.

In [ ]:
%%writefile ch5_diagnostic.py
#!/usr/bin/env python3

"""

Chapter 5 diagnostic study: how misaligned are ResNet-50 image features and

Mistral-7B caption features, and how much does a small set of paired anchors repair?



Colab (GPU runtime):

    pip install -q transformers accelerate bitsandbytes POT scikit-learn matplotlib

    python ch5_diagnostic.py extract --out feats

    python ch5_diagnostic.py analyze --feats feats --out results

CPU only, synthetic data (checks the metric code, no models needed):

    python ch5_diagnostic.py selftest



Every formula used here is derived in Appendix A of the thesis (section numbers in comments).

"""

import argparse, json, os, sys, time, zipfile, urllib.request

import numpy as np



# ----------------------------------------------------------------------------

# Metrics

# ----------------------------------------------------------------------------



def center(X):

    return X - X.mean(0, keepdims=True)



def linear_cka(X, Y):

    """Linear CKA, Equation 2.9 (derivation: Appendix A.5)."""

    X = center(np.asarray(X, np.float64)); Y = center(np.asarray(Y, np.float64))

    num = np.linalg.norm(Y.T @ X, "fro") ** 2

    den = np.linalg.norm(X.T @ X, "fro") * np.linalg.norm(Y.T @ Y, "fro")

    return float(num / den)



def cka_null_gaussian(n, d1, d2):

    """Expected CKA between unrelated isotropic Gaussians, Equation A.7 (Appendix A.5.5)."""

    return float(np.sqrt(d1 * d2 / ((n + d1 + 1) * (n + d2 + 1))))



def shuffled_cka(X, Y, n_perm, rng):

    v = [linear_cka(X, Y[rng.permutation(len(Y))]) for _ in range(n_perm)]

    return float(np.mean(v)), float(np.std(v))



class PCA:
    """Top-r principal directions of centered data (Appendix A.10)."""

    def __init__(self, r):
        self.r = r

    def fit(self, X):
        X = np.asarray(X, np.float64)
        self.mu = X.mean(0)
        _, _, Vt = np.linalg.svd(X - self.mu, full_matrices=False)
        self.W = Vt[: self.r].T
        return self

    def transform(self, X):
        return (np.asarray(X, np.float64) - self.mu) @ self.W


def procrustes_fit(X, Y):

    """Orthogonal R minimizing ||XR - Y||_F, Equation 2.5 (Appendix A.4)."""

    U, _, Vt = np.linalg.svd(X.T @ Y, full_matrices=False)

    return U @ Vt



def rel_residual(X, R, Y):

    return float(np.sum((X @ R - Y) ** 2) / np.sum(Y ** 2))



def _pdist(Z):

    sq = np.sum(Z ** 2, 1)

    D2 = np.maximum(sq[:, None] + sq[None, :] - 2 * Z @ Z.T, 0)

    return np.sqrt(D2)



def _gw2(Dx, Dy, a, G0=None, max_iter=1000):

    import ot

    kw = dict(G0=G0)

    try:

        return float(ot.gromov.gromov_wasserstein2(Dx, Dy, a, a, "square_loss", max_iter=max_iter, **kw))

    except TypeError:

        return float(ot.gromov.gromov_wasserstein2(Dx, Dy, a, a, "square_loss", numItermax=max_iter, **kw))



def gw_stats(X, Y, n, rng, max_iter=1000):

    """Gromov-Wasserstein (Equation 2.11) plus the paired-coupling cost (Equation A.9)."""

    idx = rng.choice(len(X), size=min(n, len(X)), replace=False)

    Dx, Dy = _pdist(X[idx]), _pdist(Y[idx])

    Dx = Dx / Dx.mean(); Dy = Dy / Dy.mean()

    m = len(idx); a = np.full(m, 1.0 / m)

    gw_unsup = _gw2(Dx, Dy, a, None, max_iter)                 # independent-coupling initialisation

    gw_paired_init = _gw2(Dx, Dy, a, np.eye(m) / m, max_iter)  # initialised at the known pairing

    paired = float(np.mean((Dx - Dy) ** 2))                    # Equation A.9

    perm = rng.permutation(m)

    shuffled = float(np.mean((Dx - Dy[np.ix_(perm, perm)]) ** 2))

    return dict(gw_unsup=gw_unsup, gw_paired_init=gw_paired_init, paired_cost=paired, shuffled_cost=shuffled)



# ----------------------------------------------------------------------------

# Ridge map and retrieval

# ----------------------------------------------------------------------------



def ridge_fit(X, Y, lam):

    """Equations A.11 (primal) and A.12 (dual, cheaper when k < d); Appendix A.9."""

    k, d = X.shape

    if k < d:

        A = X @ X.T; A[np.diag_indices(k)] += lam

        return X.T @ np.linalg.solve(A, Y)

    A = X.T @ X; A[np.diag_indices(d)] += lam

    return np.linalg.solve(A, X.T @ Y)



REL_LAMS = (1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0)



def ridge_cv(X, Y, rng, folds=5):

    """Pick the ridge strength using the anchors only (k-fold CV on squared error)."""

    k = len(X); folds = max(2, min(folds, k))

    scale = float(np.mean(np.sum(X ** 2, 1)))

    idx = rng.permutation(k); parts = np.array_split(idx, folds)

    best, best_err = None, np.inf

    for rel in REL_LAMS:

        lam = rel * scale; err = 0.0

        for p in parts:

            tr = np.setdiff1d(idx, p)

            W = ridge_fit(X[tr], Y[tr], lam)

            err += float(np.sum((X[p] @ W - Y[p]) ** 2))

        if err < best_err: best, best_err = lam, err

    return best



def recall_at_k(Q, C, ks=(1, 5, 10)):

    """Image-to-caption retrieval; the correct caption for query i is candidate i."""

    Qn = Q / (np.linalg.norm(Q, axis=1, keepdims=True) + 1e-12)

    Cn = C / (np.linalg.norm(C, axis=1, keepdims=True) + 1e-12)

    S = Qn @ Cn.T

    correct = np.diag(S)[:, None]

    greater = (S > correct).sum(1); equal = (S == correct).sum(1)

    rank = greater + 1 + (equal - 1) / 2.0

    return {k: float(np.mean(rank <= k)) for k in ks}



def chance_recall(n, ks=(1, 5, 10)):

    return {k: k / n for k in ks}  # Equation A.13



# ----------------------------------------------------------------------------

# Extraction (GPU)

# ----------------------------------------------------------------------------



COCO_VAL = "http://images.cocodataset.org/zips/val2017.zip"

COCO_ANN = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"



def prepare_coco(root, n_total, seed):

    os.makedirs(root, exist_ok=True)

    for url in (COCO_VAL, COCO_ANN):

        z = os.path.join(root, os.path.basename(url)); marker = z + ".done"

        if not os.path.exists(marker):

            if not os.path.exists(z):

                print("downloading", url, flush=True); urllib.request.urlretrieve(url, z)

            with zipfile.ZipFile(z) as f: f.extractall(root)

            open(marker, "w").close()

    ann = json.load(open(os.path.join(root, "annotations", "captions_val2017.json")))

    files = {i["id"]: i["file_name"] for i in ann["images"]}

    caps = {}

    for a in ann["annotations"]: caps.setdefault(a["image_id"], []).append(a["caption"].strip())

    ids = sorted(caps); assert n_total <= len(ids), f"val2017 has only {len(ids)} images"

    rng = np.random.default_rng(seed); rng.shuffle(ids); ids = ids[:n_total]

    return [(os.path.join(root, "val2017", files[i]), caps[i][int(rng.integers(len(caps[i])))]) for i in ids]



def _t(x):

    return x if hasattr(x, "shape") else x.pooler_output



def extract(a):

    import torch

    from PIL import Image

    os.makedirs(a.out, exist_ok=True)

    n_total = a.n_train + a.n_test

    items = prepare_coco(a.coco, n_total, a.seed)

    paths, caps = [p for p, _ in items], [c for _, c in items]

    dev = "cuda" if torch.cuda.is_available() else "cpu"

    print("device:", dev, flush=True)



    def images(batch_paths):
        return [Image.open(p).convert("RGB") for p in batch_paths]



# --- ResNet-50 (global-average-pooled 2048-d)

    import torchvision

    w = torchvision.models.ResNet50_Weights.DEFAULT

    rn = torchvision.models.resnet50(weights=w); rn.fc = torch.nn.Identity(); rn.eval().to(dev)

    tf = w.transforms(); feats = []

    with torch.no_grad():

        for i in range(0, n_total, a.bs):

            x = torch.stack([tf(im) for im in images(paths[i:i + a.bs])]).to(dev)

            feats.append(rn(x).float().cpu().numpy()); print("resnet", i, flush=True)

    np.save(os.path.join(a.out, "resnet.npy"), np.concatenate(feats)); del rn

    torch.cuda.empty_cache()



# --- CLIP (reference)

    from transformers import CLIPModel, CLIPProcessor

    cm = CLIPModel.from_pretrained(a.clip).to(dev).eval(); cp = CLIPProcessor.from_pretrained(a.clip)

    ci, ct = [], []

    with torch.no_grad():

        for i in range(0, n_total, a.bs):

            px = cp(images=images(paths[i:i + a.bs]), return_tensors="pt")["pixel_values"].to(dev)

            ci.append(_t(cm.get_image_features(pixel_values=px)).float().cpu().numpy())

            tk = cp(text=caps[i:i + a.bs], padding=True, truncation=True, return_tensors="pt").to(dev)

            ct.append(_t(cm.get_text_features(**tk)).float().cpu().numpy()); print("clip", i, flush=True)

    np.save(os.path.join(a.out, "clip_img.npy"), np.concatenate(ci))

    np.save(os.path.join(a.out, "clip_txt.npy"), np.concatenate(ct)); del cm

    torch.cuda.empty_cache()



# --- Mistral-7B, 4-bit; mean over caption tokens, BOS excluded; last and middle layer

    from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig

    tok = AutoTokenizer.from_pretrained(a.lm); tok.pad_token = tok.eos_token; tok.padding_side = "right"

    q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)

    lm = AutoModel.from_pretrained(a.lm, quantization_config=q, device_map="auto").eval()

    last, mid = [], []

    with torch.no_grad():

        for i in range(0, n_total, a.lm_bs):

            enc = tok(caps[i:i + a.lm_bs], return_tensors="pt", padding=True, truncation=True, max_length=64).to(lm.device)

            hs = lm(**enc, output_hidden_states=True).hidden_states

            m = enc["attention_mask"].clone(); m[:, 0] = 0          # drop the BOS position

            m = m.unsqueeze(-1).float(); den = m.sum(1).clamp(min=1)

            pool = lambda h: ((h.float() * m).sum(1) / den).cpu().numpy()

            last.append(pool(hs[-1])); mid.append(pool(hs[len(hs) // 2])); print("mistral", i, flush=True)

    np.save(os.path.join(a.out, "mistral_last.npy"), np.concatenate(last))

    np.save(os.path.join(a.out, "mistral_mid.npy"), np.concatenate(mid))



    import transformers, torch as _t_, torchvision as _tv

    json.dump(dict(seed=a.seed, n_train=a.n_train, n_test=a.n_test, lm=a.lm, clip=a.clip,

resnet="torchvision ResNet50 IMAGENET1K DEFAULT weights", dataset="COCO val2017 (5k images), one random caption per image",

versions=dict(torch=_t_.__version__, torchvision=_tv.__version__, transformers=transformers.__version__)),

              open(os.path.join(a.out, "run_config.json"), "w"), indent=2)

    json.dump([dict(image=p, caption=c) for p, c in items], open(os.path.join(a.out, "items.json"), "w"))

    print("saved features to", a.out)



# ----------------------------------------------------------------------------

# Analysis

# ----------------------------------------------------------------------------



def ms(m, s=None, pct=False):

    f = (lambda v: f"{100 * v:.1f}%") if pct else (lambda v: f"{v:.3f}")

    return f(m) if s is None else f"{f(m)} ± {f(s)}"



def zscore_fit(X):

    mu = X.mean(0); sd = X.std(0) + 1e-6

    return mu, sd



def analyze(a):

    rng = np.random.default_rng(a.seed)

    cfg = json.load(open(os.path.join(a.feats, "run_config.json")))

    ntr, nte = cfg["n_train"], cfg["n_test"]

    L = lambda f: np.load(os.path.join(a.feats, f)).astype(np.float64)

    R, Ml, Mm, CI, CT = L("resnet.npy"), L("mistral_last.npy"), L("mistral_mid.npy"), L("clip_img.npy"), L("clip_txt.npy")

    n = len(R); assert n == ntr + nte

    tr, te = slice(0, ntr), slice(ntr, n)

    os.makedirs(a.out, exist_ok=True)

    ks = [int(k) for k in a.ks.split(",")]

    res = dict(config=cfg, args=vars(a))



    def prep(X):

        """Centre (and optionally z-score) with unpaired TRAIN-set statistics, shared by every condition."""

        mu = X[tr].mean(0)

        if a.no_zscore: return X - mu

        return (X - mu) / (X[tr].std(0) + 1e-6)



    def pca_pair(X, Y):

        r = int(min(a.pca_dim, X.shape[1], Y.shape[1], ntr - 1))

        px, py = PCA(r).fit(X[tr]), PCA(r).fit(Y[tr])

        Zx, Zy = px.transform(X), py.transform(Y)

        sx = np.sqrt(np.mean(np.sum(Zx[tr] ** 2, 1))); sy = np.sqrt(np.mean(np.sum(Zy[tr] ** 2, 1)))

        return Zx / sx, Zy / sy, r



# ---------------- Table 5.1: alignment metrics

    t51 = {}

    for name, M in (("mistral_last", Ml), ("mistral_mid", Mm)):

        cka_p = linear_cka(R, M); cka_s = shuffled_cka(R, M, a.n_perm, rng)

        cka_pz = linear_cka(prep(R), prep(M)); cka_sz = shuffled_cka(prep(R), prep(M), a.n_perm, rng)

        Zx, Zy, r = pca_pair(prep(R), prep(M))

        Rm = procrustes_fit(Zx[tr], Zy[tr])

        proc_p = rel_residual(Zx[te], Rm, Zy[te])

        proc_s = float(np.mean([rel_residual(Zx[te], Rm, Zy[te][rng.permutation(nte)]) for _ in range(a.n_perm)]))

        gw = gw_stats(Zx[te], Zy[te], a.n_gw, rng)

        t51[name] = dict(cka_raw=cka_p, cka_raw_shuffled=cka_s, cka_z=cka_pz, cka_z_shuffled=cka_sz,

procrustes_paired=proc_p, procrustes_shuffled=proc_s, gw=gw, pca_dim=r)

        print(name, json.dumps(t51[name], default=float), flush=True)

    Zx, Zc, r = pca_pair(prep(R), prep(CI))

    Rm = procrustes_fit(Zx[tr], Zc[tr])

    t51["resnet_vs_clip_image"] = dict(

cka_raw=linear_cka(R, CI), cka_z=linear_cka(prep(R), prep(CI)),

cka_raw_shuffled=shuffled_cka(R, CI, a.n_perm, rng),

procrustes_paired=rel_residual(Zx[te], Rm, Zc[te]), gw=gw_stats(Zx[te], Zc[te], a.n_gw, rng), pca_dim=r)

    t51["clip_image_vs_clip_text"] = dict(cka_raw=linear_cka(CI, CT), cka_raw_shuffled=shuffled_cka(CI, CT, a.n_perm, rng))

    t51["analytic_gaussian_null_cka"] = dict(

resnet_vs_mistral=cka_null_gaussian(n, R.shape[1], Ml.shape[1]),

resnet_vs_clip_image=cka_null_gaussian(n, R.shape[1], CI.shape[1]))

    res["table_5_1"] = t51



# ---------------- Table 5.2: retrieval

    t52 = {}

    for name, M in (("mistral_last", Ml), ("mistral_mid", Mm)):

        Xp, Mp = prep(R), prep(M)

        Zx, Zy, r = pca_pair(Xp, Mp)

        out = {"chance": chance_recall(nte)}

        out["zero_shot_pca"] = recall_at_k(Zx[te], Zy[te])

        out["zero_shot_truncate"] = recall_at_k(Xp[te][:, :r], Mp[te][:, :r])

        sweeps = {"ridge": {}, "procrustes": {}}

        for k in ks + [ntr]:

            draws = 1 if k == ntr else a.draws

            rr, pp = [], []

            for _ in range(draws):

                idx = rng.choice(ntr, size=k, replace=False)

                lam = ridge_cv(Xp[tr][idx], Mp[tr][idx], rng)

                W = ridge_fit(Xp[tr][idx], Mp[tr][idx], lam)

                rr.append(recall_at_k(Xp[te] @ W, Mp[te]))

                Rk = procrustes_fit(Zx[tr][idx], Zy[tr][idx])

                pp.append(recall_at_k(Zx[te] @ Rk, Zy[te]))

            agg = lambda lst: {kk: (float(np.mean([d[kk] for d in lst])), float(np.std([d[kk] for d in lst]))) for kk in lst[0]}

            tag = "full" if k == ntr else str(k)

            sweeps["ridge"][tag] = agg(rr); sweeps["procrustes"][tag] = agg(pp)

            print(name, "k =", tag, "ridge R@10 =", sweeps["ridge"][tag][10][0], flush=True)

        out["anchors"] = sweeps

        t52[name] = out

    Cn = CI[te]; t52["clip_reference"] = recall_at_k(CI[te], CT[te])

    res["table_5_2"] = t52



    json.dump(res, open(os.path.join(a.out, "results.json"), "w"), indent=2, default=float)

    write_tables(res, a.out, ks, ntr)

    try:

        make_figures(res, a.out, ks, ntr)

    except Exception as e:

        print("figures skipped:", e)



def write_tables(res, out, ks, ntr):

    t51, t52, nte = res["table_5_1"], res["table_5_2"], res["config"]["n_test"]

    L = ["# Table 5.1. Alignment metrics\n",

         "CKA is on raw features over all examples (z-scored in brackets). Procrustes is the relative residual on held-out examples after PCA to a common dimension. "

         "GW columns: optimised distance from an independent-coupling start, and cost of the identity (paired) coupling vs a random-permutation coupling.\n",

         "| Comparison | Linear CKA | Procrustes residual | GW optimised | GW paired / shuffled cost |", "| --- | --- | --- | --- | --- |"]

    for nm, lab in (("mistral_last", "ResNet-50 vs Mistral-7B, last layer"), ("mistral_mid", "ResNet-50 vs Mistral-7B, middle layer")):

        d = t51[nm]

        L.append(f"| {lab} (paired) | {ms(d['cka_raw'])} ({ms(d['cka_z'])}) | {ms(d['procrustes_paired'])} | {ms(d['gw']['gw_unsup'])} | {ms(d['gw']['paired_cost'])} |")

        L.append(f"| {lab} (shuffled pairs) | {ms(*d['cka_raw_shuffled'])} ({ms(*d['cka_z_shuffled'])}) | {ms(d['procrustes_shuffled'])} | n/a | {ms(d['gw']['shuffled_cost'])} |")

    d = t51["resnet_vs_clip_image"]

    L.append(f"| ResNet-50 vs CLIP image encoder (same images) | {ms(d['cka_raw'])} (shuffled {ms(*d['cka_raw_shuffled'])}) | {ms(d['procrustes_paired'])} | {ms(d['gw']['gw_unsup'])} | {ms(d['gw']['paired_cost'])} |")

    d = t51["clip_image_vs_clip_text"]

    L.append(f"| CLIP image vs CLIP text (jointly trained reference) | {ms(d['cka_raw'])} (shuffled {ms(*d['cka_raw_shuffled'])}) | n/a | n/a | n/a |")

    g = t51["analytic_gaussian_null_cka"]

    L.append(f"| Analytic isotropic-Gaussian null (Appendix A.5.5) | ResNet-Mistral {ms(g['resnet_vs_mistral'])}; ResNet-CLIP {ms(g['resnet_vs_clip_image'])} | n/a | n/a | n/a |")

    L += ["", "# Table 5.2. Image-to-caption retrieval\n", f"{nte} candidate captions per query; mean ± std over anchor draws.\n"]

    for nm in ("mistral_last", "mistral_mid"):

        d = t52[nm]

        L += [f"## Caption features: {nm}\n", "| Condition | Recall@1 | Recall@5 | Recall@10 |", "| --- | --- | --- | --- |"]

        row = lambda lab, r: f"| {lab} | {ms(r[1], None, True) if not isinstance(r[1], tuple) else ms(*r[1], pct=True)} | {ms(r[5], None, True) if not isinstance(r[5], tuple) else ms(*r[5], pct=True)} | {ms(r[10], None, True) if not isinstance(r[10], tuple) else ms(*r[10], pct=True)} |"

        L.append(row("Chance", d["chance"])); L.append(row("Zero-shot, PCA to common dimension", d["zero_shot_pca"]))

        L.append(row("Zero-shot, truncate to common dimension", d["zero_shot_truncate"]))

        for method in ("ridge", "procrustes"):

            for tag in [str(k) for k in ks] + ["full"]:

                lab = f"{method} map, k = {tag if tag != 'full' else ntr}"

                L.append(row(lab, d["anchors"][method][tag]))

        L.append("")

    L.append(f"CLIP jointly trained reference: R@1 {ms(t52['clip_reference'][1], None, True)}, R@5 {ms(t52['clip_reference'][5], None, True)}, R@10 {ms(t52['clip_reference'][10], None, True)}")

    open(os.path.join(out, "tables.md"), "w").write("\n".join(L) + "\n")



def make_figures(res, out, ks, ntr):

    import matplotlib; matplotlib.use("Agg")

    import matplotlib.pyplot as plt

    BLUE, ORANGE, AQUA, GREY, INK = "#2a78d6", "#eb6834", "#1baf7a", "#8a8985", "#0b0b0b"

    plt.rcParams.update({"font.size": 10, "axes.edgecolor": GREY, "axes.labelcolor": INK, "text.color": INK,

                         "xtick.color": INK, "ytick.color": INK, "axes.spines.top": False, "axes.spines.right": False})

    t51, t52, nte = res["table_5_1"], res["table_5_2"], res["config"]["n_test"]

# Picture 9: CKA, paired vs shuffled

    rows = [("ResNet-50 vs Mistral\n(last layer)", t51["mistral_last"]["cka_raw"], t51["mistral_last"]["cka_raw_shuffled"][0]),

            ("ResNet-50 vs Mistral\n(middle layer)", t51["mistral_mid"]["cka_raw"], t51["mistral_mid"]["cka_raw_shuffled"][0]),

            ("ResNet-50 vs CLIP\nimage encoder", t51["resnet_vs_clip_image"]["cka_raw"], t51["resnet_vs_clip_image"]["cka_raw_shuffled"][0]),

            ("CLIP image vs\nCLIP text", t51["clip_image_vs_clip_text"]["cka_raw"], t51["clip_image_vs_clip_text"]["cka_raw_shuffled"][0])]

    fig, ax = plt.subplots(figsize=(7.5, 4), facecolor="#fcfcfb"); ax.set_facecolor("#fcfcfb")

    x = np.arange(len(rows)); w = 0.36

    b1 = ax.bar(x - w / 2, [r[1] for r in rows], w, color=BLUE, label="Paired")

    b2 = ax.bar(x + w / 2, [r[2] for r in rows], w, color=GREY, label="Shuffled pairs (control)")

    for bars in (b1, b2):

        for b in bars: ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.01, f"{b.get_height():.2f}", ha="center", fontsize=9)

    ax.set_xticks(x); ax.set_xticklabels([r[0] for r in rows]); ax.set_ylabel("Linear CKA"); ax.set_ylim(0, 1.05)

    ax.yaxis.grid(True, color="#e4e3df", lw=0.8); ax.set_axisbelow(True); ax.legend(frameon=False, loc="upper left")

    fig.tight_layout(); fig.savefig(os.path.join(out, "picture9_cka.png"), dpi=200); plt.close(fig)

# Picture 10: recall vs anchors

    xs = [str(k) for k in ks] + ["full"]; xp = np.arange(len(xs))

    fig, axes = plt.subplots(1, 3, figsize=(11, 3.8), facecolor="#fcfcfb", sharex=True)

    d = t52["mistral_last"]

    for ax, K in zip(axes, (1, 5, 10)):

        ax.set_facecolor("#fcfcfb")

        for method, col, lab in (("ridge", BLUE, "Ridge map"), ("procrustes", ORANGE, "Procrustes map")):

            m = np.array([d["anchors"][method][t][K][0] for t in xs]) * 100

            s = np.array([d["anchors"][method][t][K][1] for t in xs]) * 100

            ax.plot(xp, m, color=col, lw=2, marker="o", ms=5, label=lab)

            ax.fill_between(xp, m - s, m + s, color=col, alpha=0.15, lw=0)

        ax.axhline(100 * d["chance"][K], color=GREY, ls=":", lw=1.5, label="Chance")

        ax.axhline(100 * t52["clip_reference"][K], color=AQUA, ls="--", lw=1.5, label="CLIP (jointly trained)")

        ax.set_xticks(xp); ax.set_xticklabels([x if x != "full" else str(res["config"]["n_train"]) for x in xs])

        ax.set_title(f"Recall@{K}", loc="left", fontsize=10); ax.set_xlabel("Paired anchors k"); ax.set_ylabel("Recall (%)")

        ax.yaxis.grid(True, color="#e4e3df", lw=0.8); ax.set_axisbelow(True)

    axes[0].legend(frameon=False, fontsize=8, loc="upper left")

    fig.tight_layout(); fig.savefig(os.path.join(out, "picture10_retrieval.png"), dpi=200); plt.close(fig)



# ----------------------------------------------------------------------------

# Self-test on synthetic data (no GPU, no models)

# ----------------------------------------------------------------------------



def selftest():

    import tempfile

    rng = np.random.default_rng(0); ok = True

    def check(name, cond, info=""):
        nonlocal ok
        ok &= bool(cond)
        print(("PASS " if cond else "FAIL ") + name, info)



# CKA invariances (Appendix A.5.3)

    X = rng.normal(size=(300, 50)); Q, _ = np.linalg.qr(rng.normal(size=(50, 50)))

    check("CKA(X, XQ) = 1", abs(linear_cka(X, X @ Q) - 1) < 1e-9)

    check("CKA scale-invariant", abs(linear_cka(X, 3.7 * X @ Q) - 1) < 1e-9)

    Y = rng.normal(size=(300, 70))

    check("CKA in [0,1]", 0 <= linear_cka(X, Y) <= 1)

# Analytic null (Appendix A.5.5)

    A, B = rng.normal(size=(500, 200)), rng.normal(size=(500, 300))

    emp, pred = linear_cka(A, B), cka_null_gaussian(500, 200, 300)

    check("CKA null matches Equation A.7", abs(emp - pred) < 0.04, f"empirical {emp:.3f} vs predicted {pred:.3f}")

# Procrustes (Appendix A.4)

    Rt, _ = np.linalg.qr(rng.normal(size=(20, 20))); Xp = rng.normal(size=(400, 20))

    Rhat = procrustes_fit(Xp, Xp @ Rt)

    check("Procrustes recovers rotation", np.allclose(Rhat, Rt, atol=1e-8) and rel_residual(Xp, Rhat, Xp @ Rt) < 1e-15)

# GW (Appendix A.7)

    G = rng.normal(size=(120, 10)); Qg, _ = np.linalg.qr(rng.normal(size=(10, 10)))

    s = gw_stats(G, G @ Qg, 100, np.random.default_rng(1))

    check("GW paired cost \~ 0 for isometric copy", s["paired_cost"] < 1e-12, f"{s['paired_cost']:.2e}")

    check("GW paired-init \~ 0", s["gw_paired_init"] < 1e-9, f"{s['gw_paired_init']:.2e}")

    check("shuffled coupling cost >> paired", s["shuffled_cost"] > 1e-3, f"{s['shuffled_cost']:.3f}")

# Ridge primal = dual (Appendix A.9)

    Xr, Yr = rng.normal(size=(30, 80)), rng.normal(size=(30, 5)); lam = 0.7

    Wd = ridge_fit(Xr, Yr, lam); Wp = np.linalg.solve(Xr.T @ Xr + lam * np.eye(80), Xr.T @ Yr)

    check("ridge dual == primal", np.allclose(Wd, Wp, atol=1e-8))

# Chance recall (Appendix A.11)

    Qr, Cr = rng.normal(size=(1000, 16)), rng.normal(size=(1000, 16))

    r = recall_at_k(Qr, Cr, (1, 10))

    check("random retrieval near chance", r[10] < 0.03, f"R@10 = {r[10]:.3f}, chance 0.010")

# Shared-latent synthetic data: ridge retrieval works, shuffled labels do not

    n, ntr_, nte_ = 800, 600, 200

    Z = rng.normal(size=(n, 16))

    S1 = Z @ rng.normal(size=(16, 40)) + 0.1 * rng.normal(size=(n, 40))

    S2 = Z @ rng.normal(size=(16, 60)) + 0.1 * rng.normal(size=(n, 60))

    W = ridge_fit(S1[:ntr_], S2[:ntr_], 1.0)

    good = recall_at_k(S1[ntr_:] @ W, S2[ntr_:])[10]

    Wbad = ridge_fit(S1[:ntr_], S2[:ntr_][rng.permutation(ntr_)], 1.0)

    bad = recall_at_k(S1[ntr_:] @ Wbad, S2[ntr_:])[10]

    check("ridge retrieval works on shared latent", good > 0.5, f"R@10 = {good:.2f}")

    check("ridge on shuffled pairs near chance", bad < 0.15, f"R@10 = {bad:.2f} (chance 0.05)")

# End-to-end analyze on fake features

    with tempfile.TemporaryDirectory() as d:

        fe = os.path.join(d, "feats"); os.makedirs(fe)

        np.save(os.path.join(fe, "resnet.npy"), S1.astype(np.float32))

        np.save(os.path.join(fe, "mistral_last.npy"), S2.astype(np.float32))

        np.save(os.path.join(fe, "mistral_mid.npy"), (S2 @ np.eye(60)[:, ::-1]).astype(np.float32))

        np.save(os.path.join(fe, "clip_img.npy"), (Z @ rng.normal(size=(16, 32))).astype(np.float32))

        np.save(os.path.join(fe, "clip_txt.npy"), (Z @ rng.normal(size=(16, 32))).astype(np.float32))

        json.dump(dict(seed=0, n_train=ntr_, n_test=nte_), open(os.path.join(fe, "run_config.json"), "w"))

        ns = argparse.Namespace(feats=fe, out=os.path.join(d, "res"), seed=0, ks="10,50", draws=2, n_perm=3,

n_gw=60, pca_dim=16, no_zscore=False)

        analyze(ns)

        have = all(os.path.exists(os.path.join(ns.out, f)) for f in ("results.json", "tables.md", "picture9_cka.png", "picture10_retrieval.png"))

        check("analyze runs end to end and writes outputs", have)

        if have:

            rj = json.load(open(os.path.join(ns.out, "results.json")))

            full = rj["table_5_2"]["mistral_last"]["anchors"]["ridge"]["full"]["10"][0]

            zs = rj["table_5_2"]["mistral_last"]["zero_shot_pca"]["10"]

            cp, cs = rj["table_5_1"]["mistral_last"]["cka_raw"], rj["table_5_1"]["mistral_last"]["cka_raw_shuffled"][0]

            check("pipeline: ridge (all anchors) recovers shared signal", full > 0.5, f"R@10 = {full:.2f}")

            check("pipeline: zero-shot raw vectors near chance", zs < 0.15, f"R@10 = {zs:.3f}")

            check("pipeline: paired CKA exceeds shuffled control", cp > cs + 0.05, f"{cp:.3f} vs {cs:.3f}")

    print("ALL PASSED" if ok else "SOME CHECKS FAILED"); sys.exit(0 if ok else 1)



# ----------------------------------------------------------------------------



def main():

    ap = argparse.ArgumentParser(); sub = ap.add_subparsers(dest="cmd", required=True)

    e = sub.add_parser("extract")

    e.add_argument("--out", default="feats"); e.add_argument("--coco", default="coco")

    e.add_argument("--n_train", type=int, default=4000); e.add_argument("--n_test", type=int, default=1000)

    e.add_argument("--seed", type=int, default=0); e.add_argument("--bs", type=int, default=64); e.add_argument("--lm_bs", type=int, default=32)

    e.add_argument("--lm", default="mistralai/Mistral-7B-v0.1"); e.add_argument("--clip", default="openai/clip-vit-base-patch32")

    s = sub.add_parser("analyze")

    s.add_argument("--feats", default="feats"); s.add_argument("--out", default="results"); s.add_argument("--seed", type=int, default=0)

    s.add_argument("--ks", default="10,50,100,500,1000"); s.add_argument("--draws", type=int, default=5)

    s.add_argument("--n_perm", type=int, default=20); s.add_argument("--n_gw", type=int, default=500)

    s.add_argument("--pca_dim", type=int, default=256); s.add_argument("--no_zscore", action="store_true")

    sub.add_parser("selftest")

    a = ap.parse_args()

    {"extract": extract, "analyze": analyze, "selftest": lambda _: selftest()}[a.cmd](a)



if __name__ == "__main__":

    main()




Writing ch5_diagnostic.py


## 3. Optional: test the metrics without downloading models

This uses synthetic data. It is a quick check that the metric functions run.

In [ ]:
!python ch5_diagnostic.py selftest

/content/ch5_diagnostic.py:822: SyntaxWarning: invalid escape sequence '\~'
  check("GW paired cost \~ 0 for isometric copy", s["paired_cost"] < 1e-12, f"{s['paired_cost']:.2e}")
/content/ch5_diagnostic.py:824: SyntaxWarning: invalid escape sequence '\~'
  check("GW paired-init \~ 0", s["gw_paired_init"] < 1e-9, f"{s['gw_paired_init']:.2e}")
PASS CKA(X, XQ) = 1 
PASS CKA scale-invariant 
PASS CKA in [0,1] 
PASS CKA null matches Equation A.7 empirical 0.330 vs predicted 0.327
PASS Procrustes recovers rotation 
I0000 00:00:1793214370.731722     678 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
PASS GW paired cost \~ 0 for isometric copy 7.00e-19
PASS GW paired-init \~ 0 6.66e-17
PASS shuffled coupling cost >> paired 0.110
PASS ridge dual == primal 
PASS random retrieval near chance R@10 = 0.003, chance 0.010
PASS ridge retrieval works on shared latent R@10 = 1.00
PASS ridge on shuffled pairs near chance R@10 = 0.07 (chance 0.05)
mistral_last {"cka_

## 4. Extract features

Default: 4,000 training images + 1,000 test images. This downloads COCO images/annotations and model weights. For a smaller smoke test, change `--n_train` and `--n_test` to smaller values.

In [ ]:
!python ch5_diagnostic.py extract --out feats --coco coco --n_train 4000 --n_test 1000 --bs 32 --lm_bs 4

/content/ch5_diagnostic.py:822: SyntaxWarning: invalid escape sequence '\~'
  check("GW paired cost \~ 0 for isometric copy", s["paired_cost"] < 1e-12, f"{s['paired_cost']:.2e}")
/content/ch5_diagnostic.py:824: SyntaxWarning: invalid escape sequence '\~'
  check("GW paired-init \~ 0", s["gw_paired_init"] < 1e-9, f"{s['gw_paired_init']:.2e}")
device: cuda
resnet 0
resnet 32
resnet 64
resnet 96
resnet 128
resnet 160
resnet 192
resnet 224
resnet 256
resnet 288
resnet 320
resnet 352
resnet 384
resnet 416
resnet 448
resnet 480
resnet 512
resnet 544
resnet 576
resnet 608
resnet 640
resnet 672
resnet 704
resnet 736
resnet 768
resnet 800
resnet 832
resnet 864
resnet 896
resnet 928
resnet 960
resnet 992
resnet 1024
resnet 1056
resnet 1088
resnet 1120
resnet 1152
resnet 1184
resnet 1216
resnet 1248
resnet 1280
resnet 1312
resnet 1344
resnet 1376
resnet 1408
resnet 1440
resnet 1472
resnet 1504
resnet 1536
resnet 1568
resnet 1600
resnet 1632
resnet 1664
resnet 1696
resnet 1728
resnet 1760
resnet 1

## 5. Analyze features

Outputs are saved under `results/`, including `results.json`, `tables.md`, and figures. The analysis step may take a while because it runs permutations and optimal-transport calculations.

In [ ]:
!python ch5_diagnostic.py analyze --feats feats --out results --ks 10,50,100,500,1000 --draws 5 --n_perm 20 --n_gw 500 --pca_dim 256

## 6. View the generated outputs

In [ ]:
from pathlib import Path
from IPython.display import display, Image, Markdown

results_dir = Path('results')
if (results_dir / 'tables.md').exists():
    display(Markdown((results_dir / 'tables.md').read_text()))
for filename in ['picture9_cka.png', 'picture10_retrieval.png']:
    path = results_dir / filename
    if path.exists():
        display(Image(filename=str(path)))